# **Practical AI (ST1508) CA 1**

In this Jupyter Notebook, the driver dataset that have been provided will be prepared and cleaned. This will prepare the way for Data Visualisations which will be used to obtain insights. Subsequently, the data will be inserted into the SQL Data Warehouse and analysed. We will later proceed to conduct the Model Training using the Data from the Data Warehouse

---
# 1. Libraries Import and Environment Setup

Before conducting Data Preparation and Cleaning, we will firstly import the required modules and libraries. These modules and libraries are required to perform the Data Cleaning with the datasets. Subsequently, we will setup the environment so as to ensure the ETL Pipeline is consistent with every batch of data operations. This also makes the ETL Pipeline replicable.

In [ ]:
# 1) Download & unzip MySQL Connector/J
!wget -q https://dev.mysql.com/get/Downloads/Connector-J/mysql-connector-j-8.0.33.zip
!unzip -q mysql-connector-j-8.0.33.zip

# 2) Show jar path
!ls /content/mysql-connector-j-8.0.33/

In [ ]:
!pip install --no-cache-dir pyspark==3.5.1

In [ ]:
pip install mysql-connector-python

In [ ]:
"""
Cell S0 — Environment Setup (Spark)
===================================

Spark-first setup for Driver cleaning pipeline.

- Creates SparkSession
- Sets reproducibility knobs
- No pandas dependency
"""

from __future__ import annotations
import os
from typing import Any, Dict, List, Optional, Tuple
from pyspark.sql import SparkSession, DataFrame, Window
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.ml.feature import VectorAssembler, StandardScaler, PCA
from pyspark.ml import Pipeline
from pyspark.ml.linalg import DenseVector, Vectors, Matrix, DenseMatrix
import mysql.connector
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

RANDOM_STATE = 42

jar_path = "/content/mysql-connector-j-8.0.33/mysql-connector-j-8.0.33.jar"

spark = (
    SparkSession.builder
    .appName("PAI_CA1_Driver_Cleaning")
    .config("spark.jars", jar_path)
    .config("spark.driver.extraClassPath", jar_path)
    .config("spark.executor.extraClassPath", jar_path)
    .getOrCreate()
)

print("New Spark started with MySQL jar.")

spark.conf.set("spark.sql.session.timeZone", "UTC")
spark.conf.set("spark.sql.shuffle.partitions", "200")

print("Spark ready for Driver Data.")

With reference to the Code Cell above, we are able to determine the required libraries have been successfully imported and we currently have them in the local environment. We are also able to determine that the Google Drive have been mounted, implying we are able to access the provided dataset in the local environment.

---
# 2. Data Extract Layer

In this section, we will be conducting the Extraction component of the ETL Process. The Extraction process will be defined as conducting the data retrieval from the Staging Database and conducting quality checks on the Data without performing any operations yet. This step is crucial as it will allow us to start the pipeline with the latest data and allow for the cleaning of the data.

---
## 2.1 Data Extraction From Staging Database

In this sub-section, we will be extracting the Data from the Staging Database. This process is known as the Extract segment within the ETL pipeline. The data will be queried using SQL from the mySQL database and subsequently it will go through the Transform Layer after the Extraction is done successfully.

In [ ]:
"""
Cell S1 — Ingestion (DB) with Server-Side NULL Coercion & Type Casting (Spark JDBC)
"""

# --------------------------------------------------------------------
# Connection Configuration
# --------------------------------------------------------------------
db_user = os.getenv("STAGING_DB_USER", "admin")
db_pass = os.getenv("STAGING_DB_PASSWORD")
db_host = os.getenv("STAGING_DB_HOST")
db_port = int(os.getenv("STAGING_DB_PORT", "3306"))
db_name = os.getenv("STAGING_DB_NAME", "staging")
db_table = os.getenv("STAGING_DB_TABLE", "drivers_source")

jdbc_url = f"jdbc:mysql://{db_host}:{db_port}/{db_name}"

id_col = "driver_id"
cat_cols = ["driver_name", "data_of_birth", "gender", "car_brand"]
num_cols = ["no_of_years_driving_exp", "car_model_year", "driver_rating"]

def nullify_sql(col: str) -> str:
    base = f"LOWER(TRIM(`{col}`))"
    tokens = ["", "na", "n/a", "null", "nan", "."]
    wrapped = base
    for token in tokens:
        wrapped = f"NULLIF({wrapped}, '{token}')"
    return wrapped

id_expr   = f"{nullify_sql(id_col)} AS `{id_col}`"
cat_exprs = [f"{nullify_sql(c)} AS `{c}`" for c in cat_cols]
num_exprs = [f"CAST({nullify_sql(c)} AS DOUBLE) AS `{c}`" for c in num_cols]

select_cols = ",\n       ".join([id_expr] + cat_exprs + num_exprs + ["cleaned_date"])

query = f"""
SELECT
       {select_cols}
FROM `{db_table}`
WHERE cleaned_date IS NULL
"""

df_raw = (
    spark.read.format("jdbc")
    .option("url", jdbc_url)
    .option("user", db_user)
    .option("password", db_pass)
    .option("query", query)
    .load()
)

# drop artefact cols if present
for artefact_col in ["csv_index", "Unnamed: 0"]:
    if artefact_col in df_raw.columns:
        df_raw = df_raw.drop(artefact_col)

print(f"Loaded {df_raw.count():,} rows, {len(df_raw.columns)} columns.")
df_raw.show(5, truncate=False)

With reference to the Code Cell above, we are able to determine that the Extraction of the Data from the Staging Database have been successfully completed and we are able to proceed to execute the Data Quality Check operations in the next sub-section.

---
## 2.2 Data Quality Inspection

In this sub-section, we will be conducting Data Quality Inspection on the Driver Data extracted from the Staging Database. This will allow us to determine what is required for the pipeline and what we can expect to encounter when dealing with Data Abnormalities. This process is under the Extraction Phase of the ETL Process as there are no physical changes done to the Data except for extracting the Data Quality Report. The inspection will be conducted in the Code Cell below.

In [ ]:
"""
Cell S2 — Baseline Audit (Spark)
"""

def missing_profile(df: DataFrame) -> DataFrame:
    total = df.count()
    miss_exprs = [
        F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)
        for c in df.columns
    ]
    miss_row = df.agg(*miss_exprs).collect()[0].asDict()
    rows = [(c, int(miss_row[c]), (miss_row[c] / total * 100 if total else 0.0))
            for c in df.columns]
    return spark.createDataFrame(rows, ["Column Name", "Missing Count", "Missing Percentage"]) \
                .orderBy(F.desc("Missing Percentage"))

def dtype_snapshot(df: DataFrame) -> DataFrame:
    rows = [(c, t) for c, t in df.dtypes]
    return spark.createDataFrame(rows, ["Column Name", "Data Type"])

def dup_summary(df: DataFrame, ignore_cols: Optional[List[str]] = None) -> DataFrame:
    ignore_cols = ignore_cols or []
    subset_cols = [c for c in df.columns if c not in ignore_cols]
    total_rows = df.count()
    distinct_rows = df.select(*subset_cols).distinct().count()
    dupes = total_rows - distinct_rows
    pct = (dupes / total_rows * 100) if total_rows else 0.0
    status = "Duplicates Found" if dupes > 0 else "No Duplicates"
    return spark.createDataFrame(
        [(total_rows, dupes, round(pct, 4), status)],
        ["Total Rows", "Duplicate Count", "Duplicate Percentage", "Status"]
    )

audit_before_missing = missing_profile(df_raw)
audit_before_dtypes  = dtype_snapshot(df_raw)
audit_before_dups    = dup_summary(df_raw)

print("Baseline Audit BEFORE Cleaning")
audit_before_missing.show(truncate=False)
audit_before_dtypes.show(truncate=False)
audit_before_dups.show(truncate=False)

With reference to the Code Cell above, we are able to determine the various actions that needs to be done within the Transform Process of the ETL Pipeline based of the quality of the provided Driver Data.

---
# 3 Data Transform Layer

In this section, we will be conducting the Transform component of the ETL Process. The Transform process will include the various operations such as dropping duplicates and imputing missing data so as to clean and prepare the Data for the Data Warehouse. This layer will incur physical changes to the Driver Data extracted from the Staging Database previously.

---
## 3.1 Duplicate Handling

In this sub-section, we will be addressing any duplicated values within the Driver Data. Duplicated Data can cause serveral issues during Data Analysis and Machine Learning operations such as causing a change in numerical summaries despite the entry being illegitimate. Therefore, duplicated data must be handled correctly by dropping.

In [ ]:
"""
Cell S3 — Duplicate Handling (Spark)
"""

def drop_exact_duplicates(df: DataFrame, ignore_cols: Optional[List[str]] = None) -> Tuple[DataFrame, int]:
    ignore_cols = ignore_cols or []
    subset_cols = [c for c in df.columns if c not in ignore_cols]
    before = df.count()
    out = df.dropDuplicates(subset_cols)
    after = out.count()
    return out, before - after

df_dedup, n_dups = drop_exact_duplicates(df_raw, ignore_cols=["cleaned_date"])

print(f"Dropped duplicates: {n_dups:,}")
dup_summary(df_dedup, ignore_cols=["cleaned_date"]).show(truncate=False)

With reference to the Code Cell above, we are able to determine that the Duplicated Data entry have been successfully dropped.

---
## 3.2 Data Types Casting

In this sub-section, we will be defining the various Data Types for each of the feature within the Driver Data. Subsequently, we will force convert the entire feature to its respective Data Type and converting any non-compliant values to 'NaN' which will be addressed later on during the imputation of Missing Data.

In [ ]:
"""
Cell S4 — Fixed-Schema Casting (Spark)

Normalises column types for downstream processing.

Important:
    We intentionally DO NOT cast `date_of_birth` to DateType here.
    We keep it as a cleaned string so that Cell C11 can perform
    robust multi-format parsing when loading into the warehouse.
"""

ID_COL = "driver_id"

# Treated as categorical/string
CAT_COLS = [
    "driver_id",
    "driver_name",
    "gender",
    "car_brand",
]

# Treated as numeric; include both possible rating names, whichever exists
NUM_COLS = [
    "no_of_years_driving_exp",
    "car_model_year",
    "rating",
    "driver_rating",
]

# Tracked for documentation; NOT cast to date in this cell
DATE_COLS = ["date_of_birth"]

# Start from the deduplicated DF
df_cast = df_dedup

# 1) Rename data_of_birth -> date_of_birth if needed
if "data_of_birth" in df_cast.columns and "date_of_birth" not in df_cast.columns:
    df_cast = df_cast.withColumnRenamed("data_of_birth", "date_of_birth")

# 2) Keep date_of_birth as trimmed string (do NOT to_date here)
if "date_of_birth" in df_cast.columns:
    df_cast = df_cast.withColumn(
        "date_of_birth",
        F.trim(F.col("date_of_birth").cast("string")),
    )

# 3) Numeric casts
for col_name in NUM_COLS:
    if col_name in df_cast.columns:
        df_cast = df_cast.withColumn(col_name, F.col(col_name).cast("double"))

# 4) Categorical casts (string + trim)
for col_name in CAT_COLS:
    if col_name in df_cast.columns:
        df_cast = df_cast.withColumn(
            col_name,
            F.trim(F.col(col_name).cast("string")),
        )

# Optional: quick sanity check
df_cast.printSchema()


With reference to the Code Cell above, we are able to determine that the various features within the Driver Data have been casted with their respective appropriate Data Types.

---
## 3.3 Removing Invalid Numerical Values

In this section, we will be identifying and casting any invalid numerical values into NaN with reference to the business context and information derived from the Metadata. These missing values (if any) will subsequently be imputed in the next sub-section. By removing these values, it maintains the integrity and quality of the data provided.

In [ ]:
"""
Cell C6 — Rule-Based Validation (Spark)
"""

current_date = F.current_date()
current_year = F.year(F.current_date())

df_validated = df_cast

# Rule 1: date_of_birth <= today (future -> null)
if "date_of_birth" in df_validated.columns:
    df_validated = df_validated.withColumn(
        "date_of_birth",
        F.when(F.col("date_of_birth") > current_date, F.lit(None)).otherwise(F.col("date_of_birth"))
    )

# Rule 2: gender in {male, female} (case-insensitive)
if "gender" in df_validated.columns:
    df_validated = df_validated.withColumn("gender_norm", F.lower(F.trim(F.col("gender"))))
    df_validated = df_validated.withColumn(
        "gender",
        F.when(F.col("gender_norm").isin("male", "female"), F.initcap(F.col("gender_norm")))
         .otherwise(F.lit(None))
    ).drop("gender_norm")

# Rule 3: car_model_year <= current year
if "car_model_year" in df_validated.columns:
    df_validated = df_validated.withColumn(
        "car_model_year",
        F.when(F.col("car_model_year") > current_year, F.lit(None)).otherwise(F.col("car_model_year"))
    )

# Rule 4: driver_rating in [2,5]
if "driver_rating" in df_validated.columns:
    df_validated = df_validated.withColumn(
        "driver_rating",
        F.when((F.col("driver_rating") < 2) | (F.col("driver_rating") > 5), F.lit(None))
         .otherwise(F.col("driver_rating"))
    )

# Validation audit counts
def count_invalid(rule_col: str, cond) -> int:
    return df_cast.where(cond).count()

audit = {
    "invalid_dob_count": count_invalid("date_of_birth", F.col("date_of_birth") > F.current_date()) if "date_of_birth" in df_cast.columns else 0,
    "invalid_gender_count": count_invalid("gender", ~F.lower(F.col("gender")).isin("male", "female")) if "gender" in df_cast.columns else 0,
    "invalid_year_count": count_invalid("car_model_year", F.col("car_model_year") > F.year(F.current_date())) if "car_model_year" in df_cast.columns else 0,
    "invalid_rating_count": count_invalid("driver_rating", (F.col("driver_rating") < 2) | (F.col("driver_rating") > 5)) if "driver_rating" in df_cast.columns else 0
}
audit["invalid_rows_any"] = df_cast.join(df_validated, on=ID_COL, how="inner") \
    .where(
        (df_cast.date_of_birth.isNotNull() & df_validated.date_of_birth.isNull()) |
        (df_cast.gender.isNotNull() & df_validated.gender.isNull()) |
        (df_cast.car_model_year.isNotNull() & df_validated.car_model_year.isNull()) |
        (df_cast.driver_rating.isNotNull() & df_validated.driver_rating.isNull())
    ).count()

print(audit)

With reference to the Code Cell above, we are able to confirm that there are no abnormalies within the Driver Dataset provided.

---
## 3.4 Missing Data Imputation

In this sub-section, we will be imputing the Missing Data using a Machine Learning algorithm. The Machine Learning algorithm in use is and IterativeImputer with ExtraTreesRegressor. The ExtraTreesRegressor is an ensemble of randomised trees which is capable of capturing non-linear relationships and various Feature Interactions without heavy reliance on parameter tuning. This makes the algorithm very suitable for Driver Data in our case. Through using a Machine Learning algorithm to impute a data, we are not replacing the value with a 'stable value' such as mean or median, rather we hope to achieve the actual desired value that was not shown in the dataset due to multiple possible reasons.

In [ ]:
"""
Cell C7 — Auto Rule-Based Imputation (Spark-native)
"""

num_cols = ["no_of_years_driving_exp", "car_model_year", "driver_rating"]
cat_cols = ["driver_id", "driver_name", "gender", "car_brand"]

df_imputed = df_validated

# Numeric imputations
for c in num_cols:
    if c not in df_imputed.columns:
        continue

    # compute skewness (Spark has it)
    stats_row = df_imputed.select(
        F.skewness(F.col(c)).alias("skew"),
        F.avg(F.col(c)).alias("mean")
    ).collect()[0]
    skew_val = stats_row["skew"]
    mean_val = stats_row["mean"]

    # median via approxQuantile
    median_val = df_imputed.approxQuantile(c, [0.5], 0.01)[0]

    method = "mean" if (skew_val is not None and abs(skew_val) < 1.5) else "median"
    fill_val = mean_val if method == "mean" else median_val

    df_imputed = df_imputed.withColumn(
        c,
        F.when(F.col(c).isNull(), F.lit(fill_val)).otherwise(F.col(c))
    )

# Categorical -> mode
for c in cat_cols:
    if c not in df_imputed.columns:
        continue

    mode_row = df_imputed.groupBy(c).count().orderBy(F.desc("count")).first()
    mode_val = mode_row[c] if mode_row else "UNKNOWN"

    df_imputed = df_imputed.withColumn(
        c,
        F.when(F.col(c).isNull(), F.lit(mode_val)).otherwise(F.col(c))
    )

print("Imputation done.")

With reference to the Code Cell above, we are able to verify that the Missing Data have been successfully imputed and we are able to proceed with identifying the various outliers and abnormalies within our Driver Data.

---
## 3.5 Outliers Identification and Addressment

In this sub-section, we will be conducting the Outliers Identification and Addressment. Outliers can often skew Machine Learning and Data Visualisation due to their extreme value (either extremely high or extremely low). This can cause numerical summaries and the mathematical and design principal behind various Machine Learning algorithm to be obsolete. Therefore, identifying and addressing outliers are extremely important but we should not ignore the fact that there is legitimate values in the Outliers.

---
### 3.5.1 Univariate Outliers Detection

In this sub-section, we will be executing the Univariate Outlier Detection Function. The Univariate Outlier Detection will be used for each feature and each features are independent of each other. The methodology utilised for the Outlier Dectection are Inter-Quartile Range, Median Absolute Deviation and Z-Score. The relevant mathematical formulas are indicated below:

---
**Interquartile Range (IQR) Rule**

Purpose: Flags Points far from the Central 50% of the data

$$\text{IQR} = Q_3 - Q_1$$

$$\text{Lower Bound} = Q_1 - k \times \text{IQR}$$

$$\text{Upper Bound} = Q_3 + k \times \text{IQR}$$

**Legend:**
- $Q_1$ = 25th Percentile
- $Q_3$ = 75th Percentile
- $\text{IQR}$ = Interquartile range
- $k$ = IQR Multiplier (commonly 1.5 or 3.0)

---
**Median Absolute Deviation (MAD)**

Purpose: Flags Points far from the Central 50% of the data

$$\text{MAD} = \text{median}\left(|x_i - \tilde{x}|\right)$$

$$z_i = \frac{0.6745 \times (x_i - \tilde{x})}{\text{MAD}}$$

$$\text{Outlier if } |z_i| > T_{\text{MAD}}$$

**Legend:**
- $Q_1$ = 25th Percentile
- $Q_3$ = 75th Percentile
- $\text{IQR}$ = Interquartile range
- $k$ = IQR Multiplier (commonly 1.5 or 3.0)

In [ ]:
"""
Cell 3.5.1 — Univariate Outlier Flagging (Spark)
"""

def add_uni_outlier_flags(df_in: DataFrame, num_cols: List[str], iqr_factor=1.5, z_thresh=3.0) -> Tuple[DataFrame, DataFrame]:
    df = df_in
    summary_rows = []

    for c in num_cols:
        if c not in df.columns:
            continue

        # basic moments
        sk = df.select(F.skewness(c).alias("skew")).collect()[0]["skew"]
        ku = df.select(F.kurtosis(c).alias("kurt")).collect()[0]["kurt"]

        approx_gauss = (sk is not None and abs(sk) < 0.5) and (ku is not None and abs((ku + 3) - 3.0) < 1.0)
        method = "zscore" if approx_gauss else "iqr"

        if method == "zscore":
            mu, sd = df.select(F.mean(c), F.stddev(c)).first()
            flag_col = F.abs((F.col(c) - F.lit(mu)) / (F.lit(sd) + F.lit(1e-12))) > z_thresh
        else:
            q1, q3 = df.approxQuantile(c, [0.25, 0.75], 0.01)
            iqr = q3 - q1
            lo, hi = q1 - iqr_factor * iqr, q3 + iqr_factor * iqr
            flag_col = (F.col(c) < lo) | (F.col(c) > hi)

        out_col = f"{c}__is_uni_outlier"
        df = df.withColumn(out_col, F.when(F.col(c).isNull(), F.lit(False)).otherwise(flag_col))

        n_out = df.where(F.col(out_col) == True).count()
        pct_out = n_out / df.count() * 100 if df.count() else 0.0
        summary_rows.append((c, method, n_out, round(pct_out, 3)))

    summary_df = spark.createDataFrame(summary_rows, ["column", "method", "n_outliers", "pct_outliers"]) \
                      .orderBy(F.desc("pct_outliers"))
    return df, summary_df

num_cols_auto = ["no_of_years_driving_exp", "car_model_year", "driver_rating"]

df_flagged, uni_summary = add_uni_outlier_flags(df_imputed, num_cols_auto)
uni_summary.show(truncate=False)

With reference to the Code Cell above, we are able to verify the various methods undertaken to address the Outliers in the Driver Data at the Univariate level.

---
### 3.5.3 Multivariate Outlier Detection

In this sub-section, we will be executing the Multivariate Outlier Detection Function. The Multivariate Outlier Detection will be used for the entire Driver Data acorss all the various features. The methodology utilised for the Outlier Dectection are Isolation Forest, Elliptic Envelope and Local Outlier Factor. The relevant mathematical formulas are indicated below:

In [ ]:
"""
Cell C8 — Multivariate Outlier Flagging (Spark-only) using PCA Reconstruction Error

Why PCA?
- Spark ML has PCA, but not One-Class SVM.
- Reconstruction error is a standard novelty/outlier detector:
  big error => point doesn't fit the "normal" subspace.

Outputs:
- _mv_score_raw  (reconstruction error)
- _mv_score      (0–1 normalized)
- _mv_is_outlier (binary flag based on contamination quantile)
"""

# --------------------------------------------------------
# 1) Numeric columns
# --------------------------------------------------------
num_cols_auto = ["no_of_years_driving_exp", "car_model_year", "driver_rating"]
num_cols_auto = [c for c in num_cols_auto if c in df_flagged.columns]

# Drop rows where all numeric cols are null (can't vectorize)
all_null_cond = None
for c in num_cols_auto:
    cond_c = F.col(c).isNull()
    all_null_cond = cond_c if all_null_cond is None else (all_null_cond & cond_c)

df_in = df_flagged.where(~all_null_cond)

# Fill remaining nulls with median for stability
fill_map = {}
for c in num_cols_auto:
    med = df_in.approxQuantile(c, [0.5], 0.01)[0]
    fill_map[c] = med
df_in = df_in.fillna(fill_map)

# --------------------------------------------------------
# 2) Build Spark ML pipeline: Assemble -> Scale -> PCA
# --------------------------------------------------------
assembler = VectorAssembler(
    inputCols=num_cols_auto,
    outputCol="features",
    handleInvalid="skip"
)

scaler = StandardScaler(
    inputCol="features",
    outputCol="features_scaled",
    withMean=True,
    withStd=True
)

# choose k components (rule of thumb: min(d, 2) for low-d driver data)
d = len(num_cols_auto)
k = min(d, 2)

pca = PCA(
    k=k,
    inputCol="features_scaled",
    outputCol="pca_features"
)

pipe = Pipeline(stages=[assembler, scaler, pca])
pipe_model = pipe.fit(df_in)
df_pca = pipe_model.transform(df_in)

# Extract PCA model + principal components matrix
pca_model = pipe_model.stages[-1]
pc = pca_model.pc  # d x k principal components matrix

# Broadcast pc for UDF usage
pc_bc = spark.sparkContext.broadcast(pc)

# --------------------------------------------------------
# 3) Reconstruction error UDF
#    x_hat = PC * z
#    error = ||x - x_hat||^2
# --------------------------------------------------------
@F.udf(T.DoubleType())
def recon_error(x_scaled: DenseVector, z: DenseVector) -> float:
    """
    x_scaled: scaled feature vector (d,)
    z: PCA coordinates (k,)
    pc: principal components matrix (d x k)

    returns squared reconstruction error in scaled space.
    """
    pc_local: DenseMatrix = pc_bc.value
    d_local, k_local = pc_local.numRows, pc_local.numCols

    # x_hat = PC * z
    # PC stored column-major in Spark
    x_hat = [0.0] * d_local
    for col in range(k_local):
        zval = z[col]
        for row in range(d_local):
            x_hat[row] += pc_local[row, col] * zval

    # squared error
    err = 0.0
    for i in range(d_local):
        diff = x_scaled[i] - x_hat[i]
        err += diff * diff
    return float(err)

df_scored = df_pca.withColumn(
    "_mv_score_raw",
    recon_error(F.col("features_scaled"), F.col("pca_features"))
)

# --------------------------------------------------------
# 4) Normalize score to 0–1
# --------------------------------------------------------
stats = df_scored.agg(
    F.min("_mv_score_raw").alias("min_s"),
    F.max("_mv_score_raw").alias("max_s")
).collect()[0]

min_s, max_s = stats["min_s"], stats["max_s"]
denom = (max_s - min_s) if max_s != min_s else 1e-12

df_scored = df_scored.withColumn(
    "_mv_score",
    (F.col("_mv_score_raw") - F.lit(min_s)) / F.lit(denom)
)

# --------------------------------------------------------
# 5) Flag top contamination% as outliers
# --------------------------------------------------------
contamination = 0.05  # 5% outliers assumption; tune if needed

thr = df_scored.approxQuantile("_mv_score_raw", [1.0 - contamination], 0.001)[0]

df_mv_flagged = df_scored.withColumn(
    "_mv_is_outlier",
    (F.col("_mv_score_raw") >= F.lit(thr)).cast("int")
)

# --------------------------------------------------------
# 6) Summary
# --------------------------------------------------------
n_out = df_mv_flagged.where(F.col("_mv_is_outlier") == 1).count()
total = df_mv_flagged.count()

mv_summary = {
    "method": "PCA_reconstruction_error",
    "k_components": k,
    "contamination": contamination,
    "threshold_raw": float(thr),
    "n_outliers": n_out,
    "pct_outliers": round(n_out / total * 100, 3) if total else 0.0,
    "rows_total": total
}

print(mv_summary)

# Re-attach rows that were ALL-null in numeric cols (score=0, not outlier)
df_all_null_back = df_flagged.where(all_null_cond) \
    .withColumn("_mv_score_raw", F.lit(0.0)) \
    .withColumn("_mv_score", F.lit(0.0)) \
    .withColumn("_mv_is_outlier", F.lit(0))

df_mv_flagged = df_mv_flagged.unionByName(df_all_null_back, allowMissingColumns=True)

With reference to the Code Cell above, we are able to verify the various methods undertaken to address the Outliers in the Driver Data at the Multivariate level.

---
# 4. Data Load Layer

In this section, we will be loading the Data in to the Data Warehouse in the MySQL Database. This process is known as the Load process of the ETL Pipeline. We will have the ensure that the Data is competiable with the MySQL syntax and can be Loaded successfully. We will also be adding a surropgate key as per the convention used for Data Warehouses so as to maintain integrity rules. The Data Loading operation will be conducted in the Code Cell below.

In [ ]:
"""
Cell C11 — Build / Refresh Dim_Driver (with car attributes) and mark staging as cleaned

What this does
--------------
1) From df_mv_flagged, build a Dim_Driver payload including car_brand and car_model_year.
2) Drop duplicates by (driver_id, car_brand, car_model_year) within this batch.
3) Exclude any driver/car combos that already exist in Dim_Driver (uk_driver_car),
   using STRING-CASTED join keys so types always match.
4) Append only new rows to Dim_Driver.
5) Mark driver rows in the staging table as cleaned (cleaned_date = NOW()).
"""

# -------------------------------------------------
# 0) Spark config — safer date parsing
# -------------------------------------------------
spark = SparkSession.builder.getOrCreate()
spark.conf.set("spark.sql.legacy.timeParserPolicy", "CORRECTED")

# ---------- 1) Warehouse connection config ----------
warehouse_db = os.getenv("WAREHOUSE_DB_NAME", "warehouse")
warehouse_jdbc_url = f"jdbc:mysql://{db_host}:{db_port}/{warehouse_db}"

dim_driver_table = "Dim_Driver"

print(f"Warehouse JDBC URL: {warehouse_jdbc_url}")
print(f"Dim_Driver table   : {dim_driver_table}")

# ---------- 2) Build Dim_Driver payload from df_mv_flagged ----------
work = df_mv_flagged

# Handle possible column naming (date_of_birth vs data_of_birth)
dob_col_name = (
    "date_of_birth"
    if "date_of_birth" in work.columns
    else "data_of_birth"
)

# Always treat DOB as trimmed string
dob_raw = F.trim(F.col(dob_col_name).cast("string"))

# Robust DOB conversion:
#   - M/d/yyyy or MM/dd/yyyy (e.g. 3/25/1970, 10/17/1982)
#   - yyyy-MM-dd (if already cleaned)
dob_expr = F.coalesce(
    F.to_date(dob_raw, "MM/dd/yyyy"),   # e.g. 10/17/1982
    F.to_date(dob_raw, "M/d/yyyy"),     # e.g. 3/7/1970
    F.to_date(dob_raw, "yyyy-MM-dd")    # e.g. 1970-03-25
)

df_dim_driver = (
    work.select(
        F.col("driver_id").cast("int").alias("driver_id"),
        F.col("driver_name").cast("string").alias("driver_name"),

        # Robust DOB conversion
        dob_expr.alias("date_of_birth"),

        F.col("gender").cast("string").alias("gender"),
        F.col("no_of_years_driving_exp")
            .cast("int")
            .alias("no_of_years_driving_exp"),

        # Rating: source is driver_rating in df_mv_flagged
        F.col("driver_rating")
            .cast("double")
            .alias("rating"),

        # Car attributes now live inside Dim_Driver
        F.col("car_brand")
            .cast("string")
            .alias("car_brand"),
        F.col("car_model_year")
            .cast("int")
            .alias("car_model_year"),

        # Persisted outlier flags / scores
        F.col("no_of_years_driving_exp__is_uni_outlier")
            .cast("int")
            .alias("no_of_years_driving_exp__is_uni_outlier"),
        F.col("driver_rating__is_uni_outlier")
            .cast("int")
            .alias("rating__is_uni_outlier"),
        F.col("car_model_year__is_uni_outlier")
            .cast("int")
            .alias("car_model_year__is_uni_outlier"),
        F.col("_mv_score")
            .cast("double")
            .alias("mv_score"),
        F.col("_mv_is_outlier")
            .cast("int")
            .alias("mv_is_outlier"),

        F.current_timestamp().alias("date_cleaned"),
    )
    # Avoid duplicate rows inside THIS batch
    .dropDuplicates(["driver_id", "car_brand", "car_model_year"])
)

print("Sample Dim_Driver rows to be written (pre de-dup vs warehouse):")
df_dim_driver.select("driver_id", "driver_name", "date_of_birth").show(10, truncate=False)

batch_count = df_dim_driver.count()
print(f"Prepared {batch_count} Dim_Driver rows in this batch (before checking warehouse).")

# ---------- 3) Load existing driver/car combos from Dim_Driver ----------
try:
    existing_keys_df = (
        spark.read.format("jdbc")
        .option("url", warehouse_jdbc_url)
        .option("dbtable", dim_driver_table)
        .option("user", db_user)
        .option("password", db_pass)
        .load()
        .select(
            F.col("driver_id").alias("driver_id"),
            F.col("car_brand").alias("car_brand"),
            F.col("car_model_year").alias("car_model_year"),
        )
        .dropna(subset=["driver_id", "car_brand", "car_model_year"])
        .dropDuplicates()
    )
    existing_count = existing_keys_df.count()
    print(f"Loaded {existing_count} existing driver/car combos from {dim_driver_table}.")
except Exception as exc:
    print("Warning: unable to read existing Dim_Driver; assuming empty. Details:", exc)
    existing_keys_df = spark.createDataFrame(
        [],
        schema=StructType([
            StructField("driver_id", IntegerType(), True),
            StructField("car_brand", StringType(), True),
            StructField("car_model_year", IntegerType(), True),
        ]),
    )

# ---------- 4) Exclude combos that already exist (respect uk_driver_car) ----------
# Use STRING join keys on BOTH sides so types always match.
df_dim_driver_join = (
    df_dim_driver
    .withColumn("_driver_id_key", F.col("driver_id").cast("string"))
    .withColumn("_car_brand_key", F.col("car_brand").cast("string"))
    .withColumn("_car_model_year_key", F.col("car_model_year").cast("string"))
)

existing_keys_join = (
    existing_keys_df
    .withColumn("_driver_id_key", F.col("driver_id").cast("string"))
    .withColumn("_car_brand_key", F.col("car_brand").cast("string"))
    .withColumn("_car_model_year_key", F.col("car_model_year").cast("string"))
    .select("_driver_id_key", "_car_brand_key", "_car_model_year_key")
    .dropDuplicates()
)

df_new_dim_driver = (
    df_dim_driver_join.alias("n")
    .join(
        existing_keys_join.alias("e"),
        on=["_driver_id_key", "_car_brand_key", "_car_model_year_key"],
        how="left_anti",
    )
    .drop("_driver_id_key", "_car_brand_key", "_car_model_year_key")
)

n_dim_rows = df_new_dim_driver.count()
print(f"{n_dim_rows} NEW Dim_Driver rows after excluding existing uk_driver_car combos.")

if n_dim_rows > 0:
    print("Sample NEW Dim_Driver rows to be inserted:")
    df_new_dim_driver.select("driver_id", "driver_name", "date_of_birth").show(10, truncate=False)
else:
    print("No new Dim_Driver rows to insert — all combos already exist in warehouse.")

# ---------- 5) Append ONLY new rows into Dim_Driver ----------
if n_dim_rows > 0:
    (
        df_new_dim_driver.write.format("jdbc")
        .option("url", warehouse_jdbc_url)
        .option("dbtable", dim_driver_table)
        .option("user", db_user)
        .option("password", db_pass)
        .mode("append")
        .save()
    )
    print(f"Inserted {n_dim_rows} NEW rows into {dim_driver_table}.")
else:
    print("Skipped warehouse write (no new rows).")

# ---------- 6) Mark staging driver rows as cleaned (cleaned_date = NOW()) ----------
# We can safely use ALL driver_ids from df_dim_driver (batch), not only new ones;
# updating cleaned_date is idempotent.
driver_ids = [
    r["driver_id"]
    for r in df_dim_driver.select("driver_id").distinct().collect()
    if r["driver_id"] is not None
]

print(f"Collected {len(driver_ids)} driver_ids for staging update in {db_table}.")

if driver_ids:
    conn = mysql.connector.connect(
        host=db_host,
        user=db_user,
        password=db_pass,
        database=db_name,      # staging DB (e.g. 'staging')
    )
    cursor = conn.cursor()

    placeholders = ", ".join(["%s"] * len(driver_ids))
    sql = f"""
        UPDATE {db_table}
        SET cleaned_date = NOW()
        WHERE driver_id IN ({placeholders})
    """

    cursor.execute(sql, driver_ids)
    conn.commit()

    print(
        f"Updated {cursor.rowcount} rows in staging table "
        f"'{db_table}' with cleaned_date = NOW()."
    )

    cursor.close()
    conn.close()
else:
    print("No driver_ids found — staging.cleaned_date not updated.")

With reference to the Code Cell above, we are able to determine that the Data Loading have been completed and we have successfully completed the ETL Pipeline.

In [ ]:
df_mv_flagged